# Task 3: Crop Recommendation System
**Dataset:** Crop Recommendation Dataset (Kaggle — Atharva Ingle)  
**Samples:** 2200 | **Crops:** 22 | **Features:** 7  
**Goal:** Predict the best crop based on soil nutrients and weather conditions  
**Deliverable:** Trained ML Model + Recommendations

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries loaded successfully.')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('Crop_recommendation.csv')
print(f'Shape: {df.shape}')
print(f'Crops ({df["label"].nunique()}): {sorted(df["label"].unique())}')
df.head(10)

## 2. Data Quality Check

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Class Distribution ===')
print(df['label'].value_counts())
print('\n=== Statistical Summary ===')
print(df.describe().round(2))

## 3. Exploratory Data Analysis

In [ ]:
feature_names = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']

# Feature distributions
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('Feature Distributions across All Crops', fontsize=15, fontweight='bold')
palette = sns.color_palette('tab20', n_colors=22)

for idx, feat in enumerate(feature_names):
    ax = axes[idx // 4][idx % 4]
    for i, (crop, grp) in enumerate(df.groupby('label')):
        ax.hist(grp[feat], bins=15, alpha=0.4, color=palette[i], label=crop)
    ax.set_title(feat, fontweight='bold', fontsize=11)
    ax.set_xlabel(feat)
    ax.set_ylabel('Count')

axes[1][3].axis('off')
from matplotlib.patches import Patch
handles = [Patch(color=palette[i], label=c) for i, c in enumerate(sorted(df['label'].unique()))]
axes[1][3].legend(handles=handles, title='Crop', loc='center', fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig('crop_feature_distributions.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots: N, P, K per crop
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Soil Nutrient Levels by Crop', fontsize=14, fontweight='bold')
labels = {'N': 'Nitrogen', 'P': 'Phosphorus', 'K': 'Potassium'}
for ax, nutrient in zip(axes, ['N', 'P', 'K']):
    order = df.groupby('label')[nutrient].median().sort_values().index
    sns.boxplot(data=df, x='label', y=nutrient, order=order,
                palette='tab20', ax=ax, linewidth=0.8)
    ax.set_title(f'{nutrient} — {labels[nutrient]}', fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=70, labelsize=8)
plt.tight_layout()
plt.savefig('crop_nutrient_boxplots.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[feature_names].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, square=True, annot_kws={'size': 10})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('crop_correlation_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. Model Training

In [ ]:
X = df[feature_names]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

models = {
    'Random Forest':     RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, random_state=42),
    'Decision Tree':     DecisionTreeClassifier(max_depth=15, random_state=42),
    'Naive Bayes':       GaussianNB()
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = {'model': model, 'accuracy': acc, 'y_pred': y_pred}
    print(f'{name:<25}: {acc*100:.2f}%')

best_name = max(results, key=lambda k: results[k]['accuracy'])
best_model = results[best_name]['model']
print(f'\nBest Model: {best_name} ({results[best_name]["accuracy"]*100:.2f}%)')

In [ ]:
# Model comparison chart
fig, ax = plt.subplots(figsize=(8, 4))
names = list(results.keys())
accs = [results[n]['accuracy'] * 100 for n in names]
bar_colors = ['#4CAF50' if n == best_name else '#90A4AE' for n in names]
bars = ax.bar(names, accs, color=bar_colors, edgecolor='white')
ax.bar_label(bars, fmt='%.2f%%', padding=4, fontsize=10)
ax.set_ylim(80, 102)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Model Accuracy Comparison', fontweight='bold', fontsize=13)
ax.tick_params(axis='x', rotation=10)
plt.tight_layout()
plt.savefig('crop_model_comparison.png', dpi=130, bbox_inches='tight')
plt.show()

## 5. Best Model Evaluation

In [ ]:
print(f'=== {best_name} — Classification Report ===')
print(classification_report(y_test, results[best_name]['y_pred']))

In [ ]:
# Confusion matrix
fig, ax = plt.subplots(figsize=(14, 12))
cm = confusion_matrix(y_test, results[best_name]['y_pred'], labels=best_model.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=best_model.classes_, yticklabels=best_model.classes_,
            linewidths=0.3)
ax.set_title(f'{best_name} — Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()
plt.savefig('crop_confusion_matrix.png', dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_,
                            index=feature_names).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    bar_c = ['#1976D2' if v == importances.max() else '#90CAF9' for v in importances]
    bars = ax.barh(importances.index, importances.values, color=bar_c, edgecolor='white')
    ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=9)
    ax.set_xlabel('Importance Score')
    ax.set_title(f'{best_name} — Feature Importance', fontweight='bold', fontsize=13)
    plt.tight_layout()
    plt.savefig('crop_feature_importance.png', dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Most important feature: {importances.idxmax()} ({importances.max():.4f})')

## 6. Crop Recommendation Function

In [ ]:
def recommend_crop(N, P, K, temperature, humidity, ph, rainfall, model=best_model):
    """
    Recommend the best crop based on soil and weather parameters.

    Parameters:
        N           : Nitrogen content in soil (mg/kg)
        P           : Phosphorus content in soil (mg/kg)
        K           : Potassium content in soil (mg/kg)
        temperature : Temperature (°C)
        humidity    : Relative humidity (%)
        ph          : Soil pH
        rainfall    : Annual rainfall (mm)

    Returns:
        (recommended_crop: str, top3: list of (crop, confidence%))
    """
    features = pd.DataFrame(
        [[N, P, K, temperature, humidity, ph, rainfall]],
        columns=feature_names
    )
    prediction = model.predict(features)[0]
    proba = model.predict_proba(features)[0]
    top3_idx = np.argsort(proba)[::-1][:3]
    top3 = [(model.classes_[i], round(proba[i]*100, 2)) for i in top3_idx]
    return prediction, top3


# Example recommendations using real dataset value ranges
test_cases = [
    (90,  42, 43, 20.9, 82.0, 6.5, 202.9, 'Rice paddy conditions'),
    (0,   20, 35, 29.2, 90.7, 5.8, 100.0, 'Tropical fruit belt'),
    (40,  67, 79, 18.0, 16.0, 7.3,  73.0, 'Dry legume conditions'),
    (118, 46, 19, 23.7, 79.9, 6.9,  80.5, 'Cotton growing belt'),
    (20,  30, 30, 27.0, 85.0, 5.5, 150.0, 'Banana plantation'),
]

print(f'=== CROP RECOMMENDATIONS (Model: {best_name}) ===')
print(f'{"Description":<28} {"Recommended":<18} Top 3 Predictions')
print('-' * 85)
for N, P, K, T, H, ph, R, desc in test_cases:
    crop, top3 = recommend_crop(N, P, K, T, H, ph, R)
    top3_str = '  |  '.join([f"{c} {p}%" for c, p in top3])
    print(f'{desc:<28} {crop.upper():<18} {top3_str}')

## 7. Save Model

In [ ]:
joblib.dump(best_model, 'crop_recommendation_model.pkl')
print(f'Model saved: crop_recommendation_model.pkl')
print(f'Model: {best_name} | Accuracy: {results[best_name]["accuracy"]*100:.2f}%')
print(f'Trained on {len(X_train)} samples | Tested on {len(X_test)} samples')
print(f'Supports {len(best_model.classes_)} crops: {list(best_model.classes_)}')